In [1]:
#import argparse
#import time
import os
import certifi
import pandas as pd
import numpy as np
import rdflib
import sys
#from sentence_transformers import SentenceTransformer
#from sklearn.metrics.pairwise import cosine_similarity
from SPARQLWrapper import SPARQLWrapper, JSON
from utils.functions import get_node, get_subgraph
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

In [3]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

### Wikidata Query Service 

In [5]:
# SPARQL endpoint --> WIKIDATA
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

### Keyword search

In [6]:
keyword = "flood"

In [7]:
# keyword search --> returns the ID, the label and the description
node_found = get_node(keyword, sparql, JSON)

In [8]:
node_found

,node,nodeLabel,description
0,http://www.wikidata.org/entity/Q3223176,flood,overflow of water that submerges land which is...
1,http://www.wikidata.org/entity/Q8068,flood,overflow of water that submerges land


In [9]:
selected_item = ""
selected_item_label = ""

# node selection
node_number = 1

# extract the first element (column "item") of the selected row
if node_number <= len(node_found):
    selected_item = node_found.iloc[node_number]['node']
    selected_item_label = node_found.iloc[node_number]['nodeLabel']
    
print(selected_item)
print(selected_item_label)

http://www.wikidata.org/entity/Q8068
flood


### Subgraph extraction from Wikidata

In [10]:
# parameter to control the traversal depth
depth = 2

In [11]:
goal = "Anticipate the impact of floods on people"

In [12]:
model_sts = SentenceTransformer('all-mpnet-base-v2')

In [13]:
result_subgraph = get_subgraph(selected_item, goal, model_sts, sparql, JSON, depth)
result_subgraph

queue: [('http://www.wikidata.org/entity/Q8068', 1)]


current_item: http://www.wikidata.org/entity/Q8068


current_depth: 1


{'http://www.wikidata.org/entity/Q8068'}


DATA:


                                                                                                                                                                                           TRIPLE_LABELIZED                                 SUBJECT_ID                                PREDICATE_ID                                  OBJECT_ID                                       GOAL
0                                                                                                                                                                        flood subclass of natural disaster       http://www.wikidata.org/entity/Q8068    http://www.wikidata.org/prop/direct/P279       http://www.wikidata.org/entity/Q8065  Anticipate the impact of floods on people
1                                                                   

/Users/aabdouls/Desktop/goal_relaxation/utils/functions.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(highest_score_triples, ignore_index=True)



Highest scoring triples:
                                                                                                                                  TRIPLE_LABELIZED                                 SUBJECT_ID                               PREDICATE_ID                                OBJECT_ID                                       GOAL  SIMILARITY_SCORE
341                                                                             Decision-making for Living in Flood-prone Areas main subject flood  http://www.wikidata.org/entity/Q112891464   http://www.wikidata.org/prop/direct/P921     http://www.wikidata.org/entity/Q8068  Anticipate the impact of floods on people          0.770577
216                                                          The fraction of the global population at risk of floods is growing main subject flood  http://www.wikidata.org/entity/Q108063934   http://www.wikidata.org/prop/direct/P921     http://www.wikidata.org/entity/Q8068  Anticipate the impact of floo

/Users/aabdouls/Desktop/goal_relaxation/utils/functions.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(highest_score_triples, ignore_index=True)



Highest scoring triples:
                                                                                 TRIPLE_LABELIZED                                 SUBJECT_ID                               PREDICATE_ID                                  OBJECT_ID                                       GOAL  SIMILARITY_SCORE
2                              Decision-making for Living in Flood-prone Areas main subject flood  http://www.wikidata.org/entity/Q112891464   http://www.wikidata.org/prop/direct/P921       http://www.wikidata.org/entity/Q8068  Anticipate the impact of floods on people          0.770577
3                               Decision-making for Living in Flood-prone Areas main subject risk  http://www.wikidata.org/entity/Q112891464   http://www.wikidata.org/prop/direct/P921     http://www.wikidata.org/entity/Q104493  Anticipate the impact of floods on people          0.750718
10                          Decision-making for Living in Flood-prone Areas publisher OUR Archive  http://www.

/Users/aabdouls/Desktop/goal_relaxation/utils/functions.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(highest_score_triples, ignore_index=True)


... (7 outputs left out)

/Users/aabdouls/Desktop/goal_relaxation/utils/functions.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(highest_score_triples, ignore_index=True)



Highest scoring triples:
                                                                                                                                                         TRIPLE_LABELIZED                                 SUBJECT_ID                               PREDICATE_ID                                  OBJECT_ID                                       GOAL  SIMILARITY_SCORE
3                           An examination of decision making in adjustment to the flood hazard with particular reference to the lower Manawatu region main subject flood  http://www.wikidata.org/entity/Q112838694   http://www.wikidata.org/prop/direct/P921       http://www.wikidata.org/entity/Q8068  Anticipate the impact of floods on people          0.663388
8             An examination of decision making in adjustment to the flood hazard with particular reference to the lower Manawatu region publisher Massey Research Online  http://www.wikidata.org/entity/Q112838694   http://www.wikidata.org/prop/direct/P12

/Users/aabdouls/Desktop/goal_relaxation/utils/functions.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(highest_score_triples, ignore_index=True)


,TRIPLE_LABELIZED,SUBJECT_ID,PREDICATE_ID,OBJECT_ID,GOAL,SIMILARITY_SCORE
0,Decision-making for Living in Flood-prone Area...,http://www.wikidata.org/entity/Q112891464,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.770577
1,The fraction of the global population at risk ...,http://www.wikidata.org/entity/Q108063934,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.747397
2,flood topic's main category Category:Floods,http://www.wikidata.org/entity/Q8068,http://www.wikidata.org/prop/direct/P910,http://www.wikidata.org/entity/Q6457081,Anticipate the impact of floods on people,0.684963
3,Floods and the COVID ‐19 pandemic—A new double...,http://www.wikidata.org/entity/Q105551355,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.683205
4,The flooding main subject flood,http://www.wikidata.org/entity/Q120686997,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.680557
5,An examination of decision making in adjustmen...,http://www.wikidata.org/entity/Q112838694,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.663388
6,flood has effect property damage,http://www.wikidata.org/entity/Q8068,http://www.wikidata.org/prop/direct/P1542,http://www.wikidata.org/entity/Q554774,Anticipate the impact of floods on people,0.658270
7,Decision-making for Living in Flood-prone Area...,http://www.wikidata.org/entity/Q112891464,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q8068,Anticipate the impact of floods on people,0.770577
8,Decision-making for Living in Flood-prone Area...,http://www.wikidata.org/entity/Q112891464,http://www.wikidata.org/prop/direct/P921,http://www.wikidata.org/entity/Q104493,Anticipate the impact of floods on people,0.750718
9,Decision-making for Living in Flood-prone Area...,http://www.wikidata.org/entity/Q112891464,http://www.wikidata.org/prop/direct/P123,http://www.wikidata.org/entity/Q111174468,Anticipate the impact of floods on people,0.744327
